# Regularization for Boston Housing Regression

This notebook studies feature scaling and regularization for predicting Boston housing values. It builds linear and polynomial regression models, compares Ridge and Lasso penalties, and evaluates their coefficient behavior and $R^2$ performance on held-out data.

In [2]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

In [3]:
np.random.seed(72018)



def to_2d(array):
    return array.reshape(array.shape[0], -1)


    
def plot_exponential_data():
    data = np.exp(np.random.normal(size=1000))
    plt.hist(data)
    plt.show()
    return data
    
def plot_square_normal_data():
    data = np.square(np.random.normal(loc=5, size=1000))
    plt.hist(data)
    plt.show()
    return data

In [4]:
with open('boston_housing_clean.pickle', 'rb') as to_read:
    boston = pd.read_pickle(to_read)
boston_data = boston['dataframe']
boston_description = boston['description']

# show the first 5 rows using dataframe.head() method
print("The first 5 rows of the dataframe") 
boston_data.head()

The first 5 rows of the dataframe


,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0.00632,18.0,2.31,0.0,0.538,6.575,65.2,4.0900,1.0,296.0,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0.0,0.469,6.421,78.9,4.9671,2.0,242.0,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0.0,0.469,7.185,61.1,4.9671,2.0,242.0,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0.0,0.458,6.998,45.8,6.0622,3.0,222.0,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0.0,0.458,7.147,54.2,6.0622,3.0,222.0,18.7,396.90,5.33,36.2


## Load and Inspect the Dataset

The cleaned Boston Housing data is loaded from a local pickle file. `MEDV` is used as the target, while the remaining columns form the predictor matrix.

In [5]:
y_col = "MEDV"

X = boston_data.drop(y_col, axis=1)
y = boston_data[y_col]

In [6]:
from sklearn.preprocessing import StandardScaler

s = StandardScaler()
X_ss = s.fit_transform(X)

## Feature Scaling and Manual Verification

StandardScaler places each predictor on a comparable scale before regularization. A manual NumPy implementation is checked with `np.allclose` to confirm the transformation.

In [7]:
#Hint:

a = np.array([[1, 2, 3], 
              [4, 5, 6]]) 
print(a) # 2 rows, 3 columns

[[1 2 3]
 [4 5 6]]


In [8]:
a.mean(axis=0) # mean along the *columns*

array([2.5, 3.5, 4.5])

In [9]:
a.mean(axis=1) # mean along the *rows*

array([2., 5.])

In [10]:
X2 = np.array(X)
man_transform = (X2-X2.mean(axis=0))/X2.std(axis=0)
np.allclose(man_transform, X_ss)

True

In [11]:
from sklearn.linear_model import LinearRegression

In [12]:
lr = LinearRegression()

y_col = "MEDV"

X = boston_data.drop(y_col, axis=1)
y = boston_data[y_col]

## Linear Regression Coefficients

The unscaled and standardized linear models are compared through their coefficients. Standardized coefficients make relative feature influence easier to inspect because the predictors share a common scale.

In [13]:
lr.fit(X, y)
print(lr.coef_) # min = -18

[-1.07170557e-01  4.63952195e-02  2.08602395e-02  2.68856140e+00
 -1.77957587e+01  3.80475246e+00  7.51061703e-04 -1.47575880e+00
  3.05655038e-01 -1.23293463e-02 -9.53463555e-01  9.39251272e-03
 -5.25466633e-01]


In [14]:
from sklearn.preprocessing import StandardScaler

In [15]:
s = StandardScaler()
X_ss = s.fit_transform(X)

lr2 = LinearRegression()
lr2.fit(X_ss, y)
print(lr2.coef_) # coefficients now "on the same scale"

[-0.92041113  1.08098058  0.14296712  0.68220346 -2.06009246  2.67064141
  0.02112063 -3.10444805  2.65878654 -2.07589814 -2.06215593  0.85664044
 -3.74867982]


In [16]:
pd.DataFrame(zip(X.columns, lr2.coef_)).sort_values(by=1)

,0,1
12,LSTAT,-3.748680
7,DIS,-3.104448
9,TAX,-2.075898
10,PTRATIO,-2.062156
4,NOX,-2.060092
0,CRIM,-0.920411
6,AGE,0.021121
2,INDUS,0.142967
3,CHAS,0.682203
11,B,0.856640


In [17]:
from sklearn.linear_model import Lasso
from sklearn.preprocessing import PolynomialFeatures

pf = PolynomialFeatures(degree=2, include_bias=False,)
X_pf = pf.fit_transform(X)

X_pf_ss = s.fit_transform(X_pf)

## Polynomial Features and Lasso

Second-degree polynomial features expand the predictor space. Lasso is then fitted with different alpha values to show how stronger regularization reduces coefficient magnitude and can set coefficients exactly to zero.

In [18]:
las = Lasso()
las.fit(X_pf_ss, y)
las.coef_ 

array([-0.        ,  0.        , -0.        ,  0.        , -0.        ,
        0.        , -0.        , -0.        , -0.        , -0.        ,
       -0.99073506,  0.        , -0.        , -0.        ,  0.        ,
       -0.        ,  0.06751971, -0.        , -0.        , -0.        ,
       -0.        , -0.        , -0.        , -0.        , -0.        ,
       -0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        , -0.        ,  0.        ,
       -0.        , -0.        , -0.        , -0.05012402, -0.        ,
       -0.        , -0.        , -0.        , -0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        , -0.        ,
       -0.        , -0.        , -0.        , -0.        , -0.        ,
       -0.        ,  0.        , -0.        ,  3.30027884, -0.  

In [19]:
las01 = Lasso(alpha = 0.1)
las01.fit(X_pf_ss, y)
print('sum of coefficients:', abs(las01.coef_).sum() )
print('number of coefficients not equal to 0:', (las01.coef_!=0).sum())

sum of coefficients: 26.172415115426773
number of coefficients not equal to 0: 23


In [20]:
las1 = Lasso(alpha = 1)
las1.fit(X_pf_ss, y)
print('sum of coefficients:',abs(las1.coef_).sum() )
print('number of coefficients not equal to 0:',(las1.coef_!=0).sum())


sum of coefficients: 8.47240522776016
number of coefficients not equal to 0: 7


In [21]:
from sklearn.metrics import r2_score
r2_score(y,las.predict(X_pf_ss))

0.7207000461229028

In [22]:
from sklearn.model_selection import train_test_split

## Held-Out Evaluation

The polynomial features are split into training and test sets with `random_state=72018`. Scaling is fitted on the training data only, and $R^2$ is computed on the untouched test data for a more realistic comparison.

In [23]:
X_train, X_test, y_train, y_test = train_test_split(X_pf, y, test_size=0.3, 
                                                    random_state=72018)

X_train_s = s.fit_transform(X_train)
las.fit(X_train_s, y_train)
X_test_s = s.transform(X_test)
y_pred = las.predict(X_test_s)
r2_score(y_test, y_pred)

0.6780325981174931

In [24]:
X_train_s = s.fit_transform(X_train)
las01.fit(X_train_s, y_train)
X_test_s = s.transform(X_test)
y_pred = las01.predict(X_test_s)
r2_score(y_test, y_pred)

0.7999261342846061

In [25]:
# Decreasing regularization and ensuring convergence
las001 = Lasso(alpha = 0.001, max_iter=100000)

# Transforming training set to get standardized units
X_train_s = s.fit_transform(X_train)

# Fitting model to training set
las001.fit(X_train_s, y_train)

# Transforming test set using the parameters defined from training set
X_test_s = s.transform(X_test)

# Finding prediction on test set
y_pred = las001.predict(X_test_s)

# Calculating r2 score
print("r2 score for alpha = 0.001:", r2_score(y_test, y_pred))


# Part 2

# Using vanilla Linear Regression
lr = LinearRegression()

# Fitting model to training set
lr.fit(X_train_s, y_train)

# predicting on test set
y_pred_lr = lr.predict(X_test_s)

# Calculating r2 score
print("r2 score for Linear Regression:", r2_score(y_test,y_pred_lr))


# Part 3
print('Magnitude of Lasso coefficients:', abs(las001.coef_).sum())
print('Number of coeffients not equal to 0 for Lasso:', (las001.coef_!=0).sum())

print('Magnitude of Linear Regression coefficients:', abs(lr.coef_).sum())
print('Number of coeffients not equal to 0 for Linear Regression:', (lr.coef_!=0).sum())
### END SOLUTION

r2 score for alpha = 0.001: 0.884789323687439
r2 score for Linear Regression: 0.8689110469231076
Magnitude of Lasso coefficients: 435.5723229043836
Number of coeffients not equal to 0 for Lasso: 90
Magnitude of Linear Regression coefficients: 1183.8918138675817
Number of coeffients not equal to 0 for Linear Regression: 104


In [26]:
from sklearn.linear_model import Ridge

# Decreasing regularization and ensuring convergence

r = Ridge(alpha = 0.001)
X_train_s = s.fit_transform(X_train)
r.fit(X_train_s, y_train)
X_test_s = s.transform(X_test)
y_pred_r = r.predict(X_test_s)

# Calculating r2 score
r.coef_

array([  7.69158031,   9.88890473, -25.00525751,   5.31046818,
        -2.60993702,  14.95803313,  22.31375473, -22.8579177 ,
        27.76499232,  -1.56512089,  17.07749893,  21.85840549,
        11.57430607,   1.0501267 ,   0.43058525,  13.78571186,
         1.8138621 ,  -8.34497303,   4.99428114,  -3.56428247,
        -3.43620979, -16.33550823,  -7.04652168,   6.60858547,
        -1.47661   ,   4.68585754,  -1.30404236,  -0.05944784,
        -0.30379373, -12.84226172,   1.97669113,   1.08130677,
        -0.67550448,  -1.07931934,   4.49162494,  -4.26689794,
         4.67486611,  -1.28089201,   8.66602804,  -0.27373083,
        -8.11960573,  11.78297941,   6.53845776,   1.32299881,
         2.05956326,   0.89895643,   1.7894665 ,   4.74421976,
        -4.66441531,   5.31046818,  -3.23646667,  -8.66803392,
         0.97293208,   1.13586914,   0.29037187,  -1.63142731,
        -2.92599804,   2.92315095,  -0.73510358,  11.89603615,
         0.75403388,  -7.53119729,  18.30567198, -22.16

## Ridge and Final Comparisons

Ridge regression is evaluated alongside Lasso by comparing coefficient magnitude, sparsity, and $R^2$. The notebook closes with a baseline linear model on standardized original features to distinguish the effect of polynomial expansion from the effect of regularization.

In [27]:
las001.coef_

array([ 0.00000000e+00,  0.00000000e+00, -1.70077961e+01,  2.59218903e+00,
        0.00000000e+00,  1.34310751e+01,  1.00617885e+01, -1.95420304e+01,
        9.23745604e+00,  0.00000000e+00,  6.15850148e+00,  1.70865644e+01,
        1.14862999e+01,  1.20761498e+00,  2.19212098e-01,  1.08266235e+01,
        2.18532977e+00, -7.10563499e+00,  4.31223434e+00, -1.52608936e+00,
       -2.09150369e+00, -9.66889138e+00,  0.00000000e+00,  0.00000000e+00,
       -1.17685620e+00,  3.86518537e+00,  3.77856352e-01,  1.94047899e-01,
       -2.95878630e-01, -3.47818534e+00,  2.91171870e-01,  7.20249225e-01,
       -7.95193179e-01, -7.39638413e-01,  2.40656329e+00, -8.91898909e-01,
        2.83533285e+00, -9.83613898e-01,  3.86402286e+00, -9.62972947e-01,
        6.91383535e+00,  6.19985710e+00,  4.19727921e+00,  8.92131589e-01,
        2.02792608e+00,  2.60830224e+00, -3.97491024e+00,  2.57367502e+00,
       -4.56754935e+00,  2.08634607e+00, -2.00046119e+00, -7.41198425e+00,
        1.60411709e+00,  

In [28]:
print(np.sum(np.abs(r.coef_)))
print(np.sum(np.abs(las001.coef_)))

print(np.sum(r.coef_ != 0))
print(np.sum(las001.coef_ != 0))

792.8673754966087
435.5723229043836
104
90


In [29]:
y_pred = r.predict(X_pf_ss)
print(r2_score(y, y_pred))

y_pred = las001.predict(X_pf_ss)
print(r2_score(y, y_pred))

0.9076091395031932
0.910350344203442


In [30]:
X_train, X_test, y_train, y_test = train_test_split(X_ss, y, test_size=0.3, 
                                                    random_state=72018)

In [31]:
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)
r2_score(y_test, y_pred)

0.6982083583132745

In [32]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, 
                                                    random_state=72018)

In [33]:
s = StandardScaler()
lr_s = LinearRegression()
X_train_s = s.fit_transform(X_train)
lr_s.fit(X_train_s, y_train)
X_test_s = s.transform(X_test)
y_pred_s = lr_s.predict(X_test_s)
r2_score(y_test, y_pred)

0.6982083583132745